# M_01 — Construction du graphe topologique

Ce notebook lit les **graphes topologiques propres** produits par `build_topology.py` (`04_topo/topo_{source}.gpkg`) et en tire les tables pivot pour le matching (M_02).

La construction de la topologie (nettoyage du Cadastre par reconstruction de couverture, détection des arcs partagés) est faite **en amont** dans `build_topology.py`. Ce notebook ne fait que **consommer** le résultat : il n'y a plus de parsing OSM ni de re-dérivation fragile.

## Distinction fondamentale

| | Sommets bruts | Nœuds topologiques |
|---|---|---|
| Définition | Tous les points du tracé | Jonctions uniquement (point triple, raccords) |
| Nombre (zone test) | centaines | 4 par source |
| Usage | densité du tracé | entrée pour en-matching |

Un **nœud topologique** = une jonction (où une frontière partagée commence/se termine).  
Un **arc** = la séquence de sommets entre deux nœuds topologiques.

## Repère important
Les positions sont en **EPSG:2154 (Lambert-93), en mètres** — directement exploitables pour mesurer les distances inter-sources et calibrer le cut-off `c`.

## Sorties
- `nodes_df` — `{source, node_id, x, y}`
- `edges_gdf` — `{source, edge_id, node_start, node_end, n_vertices, commune_a, commune_b, geometry}`
- Fichiers : `nodes.csv`, `edges.gpkg`

In [6]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

## 1. Lecture des graphes topologiques propres

On lit la couche `arcs` de chaque GeoPackage. On ne garde que les **arcs partagés** (`partage == True`, c.-à-d. `comm_b` renseigné) : ce sont les frontières inter-communales, les seules à apparier entre sources. Les nœuds topologiques sont les extrémités de ces arcs.

In [7]:
TOPO_DIR = 'C:/Users/rocrom/conversion_osm/04_topo_d031'
SOURCES  = ['bdtopo', 'cadastre', 'geofla']
PREC = 3  # arrondi (mm) pour identifier les noeuds coincidents au sein d'une source

def key(xy):
    return (round(xy[0], PREC), round(xy[1], PREC))

graphs = {}
for src in SOURCES:
    arcs = gpd.read_file(f'{TOPO_DIR}/topo_{src}.gpkg', layer='arcs')
    arcs = arcs[arcs['partage']].reset_index(drop=True)  # frontieres partagees seulement

    # noeuds topologiques = extremites des arcs, indexes par coordonnee
    node_id = {}
    node_xy = {}
    def get_id(xy):
        k = key(xy)
        if k not in node_id:
            nid = len(node_id)
            node_id[k] = nid
            node_xy[nid] = (xy[0], xy[1])
        return node_id[k]

    edges = []
    for _, row in arcs.iterrows():
        coords = list(row.geometry.coords)
        n_start = get_id(coords[0])
        n_end   = get_id(coords[-1])
        edges.append({
            'node_start': n_start,
            'node_end'  : n_end,
            'n_vertices': len(coords),
            'commune_a' : row['comm_a'],
            'commune_b' : row['comm_b'],
            'geometry'  : row.geometry,
        })

    graphs[src] = {'nodes': node_xy, 'edges': edges, 'crs': arcs.crs}

    print(f'\n=== {src} ===')
    print(f'  Noeuds topologiques : {len(node_xy)}')
    print(f'  Arcs partages       : {len(edges)}')
    for e in edges:
        print(f"    {e['commune_a']} <-> {e['commune_b']} : {e['n_vertices']} sommets")


=== bdtopo ===
  Noeuds topologiques : 1158
  Arcs partages       : 1570
    31001 <-> 31152 : 63 sommets
    31001 <-> 31239 : 237 sommets
    31152 <-> 31239 : 42 sommets
    31037 <-> 31054 : 278 sommets
    31037 <-> 31368 : 12 sommets
    31054 <-> 31368 : 24 sommets
    31320 <-> 31361 : 33 sommets
    31320 <-> 31379 : 266 sommets
    31361 <-> 31379 : 43 sommets
    31025 <-> 31340 : 20 sommets
    31025 <-> 31448 : 142 sommets
    31340 <-> 31448 : 83 sommets
    31217 <-> 31369 : 14 sommets
    31217 <-> 31535 : 16 sommets
    31369 <-> 31535 : 12 sommets
    31310 <-> 31393 : 94 sommets
    31310 <-> 31453 : 93 sommets
    31393 <-> 31453 : 110 sommets
    31121 <-> 31170 : 262 sommets
    31121 <-> 31363 : 254 sommets
    31170 <-> 31363 : 58 sommets
    31144 <-> 31548 : 172 sommets
    31236 <-> 31431 : 46 sommets
    31236 <-> 31342 : 79 sommets
    31342 <-> 31431 : 23 sommets
    31137 <-> 31558 : 28 sommets
    31137 <-> 31485 : 47 sommets
    31485 <-> 31558 : 51 so

## 2. Tables pivot

- `nodes_df` : un nœud topologique par ligne (`source`, `node_id`, `x`, `y` en mètres)
- `edges_gdf` : un arc partagé par ligne, avec la géométrie complète (LineString, EPSG:2154)

In [8]:
all_nodes, all_edges = [], []
crs_ref = None

for src, g in graphs.items():
    crs_ref = g['crs']
    for nid, (x, y) in g['nodes'].items():
        all_nodes.append({'source': src, 'node_id': nid, 'x': x, 'y': y})
    for i, e in enumerate(g['edges']):
        all_edges.append({
            'source'    : src,
            'edge_id'   : f'{src}_{i}',
            'node_start': e['node_start'],
            'node_end'  : e['node_end'],
            'n_vertices': e['n_vertices'],
            'commune_a' : e['commune_a'],
            'commune_b' : e['commune_b'],
            'geometry'  : e['geometry'],
        })

nodes_df  = pd.DataFrame(all_nodes)
edges_gdf = gpd.GeoDataFrame(all_edges, geometry='geometry', crs=crs_ref)

print('=== nodes_df ===')
print(nodes_df.to_string(index=False))
print(f'\n=== edges_gdf ({len(edges_gdf)} arcs) ===')
print(edges_gdf.drop(columns='geometry').to_string(index=False))

=== nodes_df ===
  source  node_id             x            y
  bdtopo        0 526057.700000 6.253484e+06
  bdtopo        1 527711.200000 6.253404e+06
  bdtopo        2 526951.600000 6.255986e+06
  bdtopo        3 525761.000000 6.252854e+06
  bdtopo        4 597015.900000 6.252351e+06
  bdtopo        5 598532.900000 6.250854e+06
  bdtopo        6 596945.100000 6.252537e+06
  bdtopo        7 595917.600000 6.250888e+06
  bdtopo        8 561258.500000 6.248354e+06
  bdtopo        9 561180.100000 6.248723e+06
  bdtopo       10 561210.300000 6.245323e+06
  bdtopo       11 562407.900000 6.248228e+06
  bdtopo       12 575678.200000 6.267850e+06
  bdtopo       13 575303.900000 6.267931e+06
  bdtopo       14 576326.000000 6.265471e+06
  bdtopo       15 577420.400000 6.268012e+06
  bdtopo       16 508500.100000 6.214009e+06
  bdtopo       17 508687.400000 6.213116e+06
  bdtopo       18 510196.900000 6.215059e+06
  bdtopo       19 507752.100000 6.214282e+06
  bdtopo       20 602229.300000 6.2592

## 4. Sauvegarde

In [10]:
MATCHING_DIR = 'C:/Users/rocrom/Resolving-Conflicts-in-Heterogeneous-Data--CRH_Framework/final'

nodes_df.to_csv(MATCHING_DIR + '/nodes.csv', index=False)
edges_gdf.to_file(MATCHING_DIR + '/edges.gpkg', driver='GPKG')

print('Sauvegarde :')
print(f'  nodes.csv  - {len(nodes_df)} lignes')
print(f'  edges.gpkg - {len(edges_gdf)} lignes')

Sauvegarde :
  nodes.csv  - 3550 lignes
  edges.gpkg - 4826 lignes


## 5. Bilan

Tables pivot prêtes pour M_02 :
- `nodes.csv` : positions (en mètres) des nœuds topologiques par source
- `edges.gpkg` : géométries des arcs par source

